# House Graph Generator Using GraphRAG and Neo4j
## Part 1: Creating the Graph Corpus using Neo4j

In [2]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed libraries

In [ ]:
from topologicpy.Neo4j import Neo4j
from topologicpy.Vertex import Vertex
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.GraphDB import GraphDB
from topologicpy.Neo4j import Neo4j

## 2. Check the TopologicPy version

In [5]:
print("This tutorial requires topologicpy version 0.9.29 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.29 or newer.
The version that you are using (0.9.29) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [6]:
renderer = "vscode"

## 4. Utility function to find files at a path

In [7]:
from pathlib import Path

def file_paths_in_folder(folder_path, recursive=False):
    folder = Path(folder_path)

    if not folder.is_dir():
        raise ValueError(f"Not a valid folder: {folder_path}")

    pattern = "**/*" if recursive else "*"

    return [
        str(path)
        for path in folder.glob(pattern)
        if path.is_file()
    ]


## 5. Set the path to the JSON graphs folder

In [8]:
folder_path = r"C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\access-sample_01"

## 6. Import the JSON graphs to TopologicPy

In [9]:
# There are 7 room types, stored numerically as 0-6. These correspond to living room (0), bedroom (1), bathroom (2), kitchen (3), window (4), door (5), and front door (6), respectively.

room_mapping = { 0: "living room",
                 1: "bedroom",
                 2: "bathroom",
                 3: "kitchen",
                 4: "window",
                 5: "door",
                 6: "front door"
               }
graphs = []
file_paths = file_paths_in_folder(folder_path, recursive=False)
for i, file_path in enumerate(file_paths):
    g = Graph.ByJSONPath(file_path)
    graph_dict = Topology.Dictionary(g)
    graph_id = "graph_"+(str(i).zfill(3))
    bedrooms = Dictionary.ValueAtKey(graph_dict, "bedrooms")
    graph_dict = Dictionary.SetValuesAtKeys(graph_dict, ["graph_id", "label"], [graph_id, bedrooms])
    g = Topology.SetDictionary(g, graph_dict)
    verts = Graph.Vertices(g)
    for j, v in enumerate(verts):
        vertex_id = graph_id+"_vertex_"+(str(j).zfill(4))
        d = Topology.Dictionary(v)
        d = Dictionary.SetValuesAtKeys(d, ["vertex_id", "vertex_size", "label"], [vertex_id, 15, room_mapping.get(Dictionary.ValueAtKey(d, "type"), "Unknown")])
        v = Topology.SetDictionary(v, d)
    
    graphs.append(g)


## 7. Do a visual check

In [10]:
for i in range(38,40,1):
    graph = graphs[i]
    print(graph)
    print(file_paths[i])
    Topology.Show(graph,
                vertexSizeKey="vertex_size",
                showVertexLabel=True,
                vertexLabelKey="label",
                backgroundColor="white",
                camera = [0,0,5],
                renderer=renderer)

C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\access-sample_01\accessibility_graph_5729.json


C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\access-sample_01\accessibility_graph_5821.json


In [11]:
verts = Graph.Vertices(graphs[-1])
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

['area', 'compactness', 'geometry', 'id', 'label', 'perimeter', 'type', 'vertex_color', 'vertex_id', 'vertex_size'] [18.16, 0.78, [[123.224963, 145.845173, 0.0], [123.224963, 141.066057, 0.0], [119.425302, 141.066057, 0.0], [119.425302, 145.845173, 0.0]], 'Vertex_0018', 'bedroom', 17.16, 1, '#20b2aa', 'graph_049_vertex_0000', 15]
['area', 'compactness', 'geometry', 'id', 'label', 'perimeter', 'type', 'vertex_color', 'vertex_id', 'vertex_size'] [5.5, 0.77, [[114.885609, 141.066057, 0.0], [112.87695, 141.066057, 0.0], [112.87695, 143.802311, 0.0], [114.885609, 143.802311, 0.0]], 'Vertex_0011', 'bathroom', 9.49, 2, '#ff8c00', 'graph_049_vertex_0001', 15]
['area', 'compactness', 'geometry', 'id', 'label', 'perimeter', 'type', 'vertex_color', 'vertex_id', 'vertex_size'] [1.79, 0.67, [[115.126585, 144.602976, 0.0], [114.846741, 144.602976, 0.0], [114.846741, 145.498477, 0.0], [115.126585, 145.498477, 0.0]], 'Vertex_0005', 'door', 5.79, 5, '#ff0000', 'graph_049_vertex_0002', 15]
['area', 'com

## 8. Connect to your Neo4j instance. Make sure it is running first:
* If using the Desktop version, open Neo4j Desktop
* If using the online version (Aura), go to your console at: https://console.neo4j.io/

In [ ]:
from getpass import getpass

url = input("URL: ")
username = input("Username: ")
password = getpass("Password: ")

gdb_manager = Neo4j.Manager(url=url, username=username, password=password, database=None, silent=False)
print(gdb_manager)

In [13]:
# -----------------------------------------------------------------------------
# 1. Create / connect to Graph DB
# -----------------------------------------------------------------------------

graphdb = GraphDB.ByParameters(
    provider="neo4j",
    manager=gdb_manager
)

# Clear test DB
ok = GraphDB.EmptyDatabase(graphdb)
print("EmptyDatabase:", ok)
# -----------------------------------------------------------------------------
# 2. Ensure schema
# -----------------------------------------------------------------------------

ok = GraphDB.EnsureSchema(graphdb)
print("EnsureSchema:", ok)

EmptyDatabase: True
EnsureSchema: True


In [14]:
# -----------------------------------------------------------------------------
# 4. Upsert graphs into GraphDB
# -----------------------------------------------------------------------------

total = len(graphs)
for i, g in enumerate(graphs):
    graph_id = GraphDB.UpsertGraph(graphdb = graphdb,
                                   graph = g,
                                   graphIDKey = "graph_id",
                                   vertexIDKey = "id",
                                   vertexLabelKey = "label",
                                   defaultVertexLabel = "Node",
                                   vertexCategoryKey = "category",
                                   defaultVertexCategory = "Node",
                                   edgeLabelKey = "label",
                                   defaultEdgeLabel = "CONNECTED_TO",
                                   edgeCategoryKey = "category",
                                   defaultEdgeCategory = "Edge",
                                   bidirectional = True,
                                   overwrite = False,
                                   mantissa = 6,
                                   silent = False
                                   )
    print(f"{i+1}/{total}: {graph_id}")

print("Done Upserting Graphs.")

1/50: graph_000
2/50: graph_001
3/50: graph_002
4/50: graph_003
5/50: graph_004
6/50: graph_005
7/50: graph_006
8/50: graph_007
9/50: graph_008
10/50: graph_009
11/50: graph_010
12/50: graph_011
13/50: graph_012
14/50: graph_013
15/50: graph_014
16/50: graph_015
17/50: graph_016
18/50: graph_017
19/50: graph_018
20/50: graph_019
21/50: graph_020
22/50: graph_021
23/50: graph_022
24/50: graph_023
25/50: graph_024
26/50: graph_025
27/50: graph_026
28/50: graph_027
29/50: graph_028
30/50: graph_029
31/50: graph_030
32/50: graph_031
33/50: graph_032
34/50: graph_033
35/50: graph_034
36/50: graph_035
37/50: graph_036
38/50: graph_037
39/50: graph_038
40/50: graph_039
41/50: graph_040
42/50: graph_041
43/50: graph_042
44/50: graph_043
45/50: graph_044
46/50: graph_045
47/50: graph_046
48/50: graph_047
49/50: graph_048
50/50: graph_049


In [15]:
# -----------------------------------------------------------------------------
# 5. Retrieve graph by ID
# -----------------------------------------------------------------------------

g2 = GraphDB.GraphByID(graphdb, graph_id)

print("Graph retrieved:", g2 is not None)

# -----------------------------------------------------------------------------
# 6. Corpus analytics tests
# -----------------------------------------------------------------------------

pairs = GraphDB.FetchAllPairs(graphdb)
print("Pairs:")
for row in pairs[:10]:
    print(row)

candidate_counts = GraphDB.CandidateCountsForLabels(
    graphdb,
    ["living room"]
)

print("\nCandidate Counts:")
for row in candidate_counts[:10]:
    print(row)

max_neighbors = GraphDB.MaxNeighborsForLabel(
    graphdb,
    "door"
)

print("\nMax Neighbors for door:")
print(max_neighbors)

max_neighbors = GraphDB.MaxNeighborsForLabel(
    graphdb,
    "kitchen"
)

print("\nMax Neighbors for kitchen:")
print(max_neighbors)

max_neighbors = GraphDB.MaxNeighborsForLabel(
    graphdb,
    "bathroom"
)

print("\nMax Neighbors for bathroom:")
print(max_neighbors)



example = GraphDB.FindBestExampleForLabel(
    graphdb,
    "kitchen"
)

print("\nBest Example for Kitchen:")
print(example)

# -----------------------------------------------------------------------------
# 7. List stored graphs
# -----------------------------------------------------------------------------

graphs = GraphDB.ListGraphs(graphdb)

print("\nStored Graphs:")
for g in graphs:
    print(g)

Graph retrieved: True
Pairs:
{'a_label': 'bedroom', 'b_label': 'door', 'pair': ['bedroom', 'door'], 'count': 378}
{'a_label': 'door', 'b_label': 'living room', 'pair': ['door', 'living room'], 'count': 374}
{'a_label': 'bathroom', 'b_label': 'door', 'pair': ['bathroom', 'door'], 'count': 224}
{'a_label': 'front door', 'b_label': 'living room', 'pair': ['front door', 'living room'], 'count': 100}
{'a_label': 'kitchen', 'b_label': 'living room', 'pair': ['kitchen', 'living room'], 'count': 88}
{'a_label': 'door', 'b_label': 'kitchen', 'pair': ['door', 'kitchen'], 'count': 12}
{'a_label': 'bathroom', 'b_label': 'bedroom', 'pair': ['bathroom', 'bedroom'], 'count': 10}
{'a_label': 'bathroom', 'b_label': 'living room', 'pair': ['bathroom', 'living room'], 'count': 10}
{'a_label': 'bathroom', 'b_label': 'bathroom', 'pair': ['bathroom', 'bathroom'], 'count': 4}
{'a_label': 'bedroom', 'b_label': 'living room', 'pair': ['bedroom', 'living room'], 'count': 2}

Candidate Counts:
{'label': 'door', 